# Protein-protein binder design with RFdiffusion
### <span style="color: gray;">How to design proteins that bind another protein.</span>

In this tutorial, we'll demonstrate how to use the OpenProtein.AI Python client to
design a protein that binds another protein. We'll refer to the designed protein as the
**binder** and the protein being bound as the **target**.

<img src="../_static/walkthroughs/protein_protein_binder_design/design_problem.png" style="width: 500px;"/>

The design process consists of four main steps:

1. **Query Specification**: Specify the design problem as a "query", including

    1. the target protein
    1. the specific epitope that we want to target
    1. the length of binder

1. **Structure Generation**: Generate plausible structures for the binder, using
   RFdiffusion.

1. **Sequence Design**: Generate corresponding sequences for each binder structure,
   using ProteinMPNN.

1. **In Silico Validation**: Validate the designed sequences by predicting their
   structures. Rank them based on structure prediction metrics to select the best
   candidates.

## Prerequisites

To run this tutorial, you'll need a Python environment containing the following
packages:

- `openprotein_python>=0.9.1`
- `molviewspec` (for structure visualization)

Additionally, you should have your credentials set up in `~/.openprotein/config.toml` to
authenticate with the OpenProtein.AI API. Below, we

- import the necessary packages,
- create a `data` directory containing artifacts used and created by this notebook, and 
- connect to the OpenProtein.AI API

In [ ]:
import itertools
from dataclasses import dataclass
from pathlib import Path

import numpy as np

import molviewspec as mvs
from molviewspec.nodes import RepresentationTypeT

import openprotein
from openprotein.sequence import get_intervals
from openprotein import Protein, Model

DATA_DIR = Path("data/")
DATA_DIR.mkdir(parents=True, exist_ok=True)

session = openprotein.connect()
print("✅ Successfully connected to the OpenProtein.AI API!")

## Step 1: Query Specification
### <span style="color: gray;">Specify the protein-protein binder design problem</span>

In this tutorial, we focus on designing a binder for the **Interleukin-7 receptor alpha 
(IL-7Rα)**, a key target in the human immune system. This specific design problem is 
adapted from the original **RFdiffusion** study ([Watson et al., 2023](https://www.nature.com/articles/s41586-023-06415-8)),
which demonstrated the de novo design of high-affinity binders to this receptor.

In this step, we will create an object, which we refer to as the **query**, that
represents this design problem. The query will be a `MolecularComplex` object containing
one `Protein` chain representing the target, and one `Protein` chain representing the
binder. We will use these classes (`MolecularComplex` and `Protein`) to create a query that specifies:

1. **Target:** The sequence and structure of the IL-7Rα extracellular domain.
1. **Epitope:** The specific residues of IL-7Rα where the binder should bind.
1. **Binder Length:** The desired length of the de novo protein.

## Step 1.1: Specify the target

For the IL-7Rα target, we use the structure from RCSB PDB entry 3DI3. We use the helper
method `Protein.from_pdb_id` to download and parse the structure into a `Protein`
object, which provides a convenient interface for handling the sequence and structure
of a single protein chain.

In [ ]:
# Load the IL-7Rα chain, which is chain B
target = Protein.from_pdb_id(pdb_id="3DI3", chain_id="B")
print("target name:", target.name)
print("target sequence:", target.sequence)
print("target length:", len(target))

target sequence: b'GSHMESGYAQNGDLEDAELDDYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIETKKFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDENKWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTPEINNSSGEMD'
target coordinates shape: (223, 37, 3)
target plddt shape: (223,)
target name: 3DI3


Before we continue, we note that although our `target` protein has 223 residues, the
structure at some of these residues is *unspecified*. That is, for some of the residues,
the coordinates of their atoms is unknown.

To see which residues have unspecified structure, we can use the method
`Protein.get_structure_mask` to get a boolean array that indicates, at each residue
position, whether the structure is unspecified. Since this boolean array tends to be
quite long, we can summarize it using the function `get_intervals`, which will tell us
which intervals in the array are `True` (structure unspecified), and which intervals are
False (structure specified).

In [ ]:
structure_mask = target.get_structure_mask()
print("first 10 values of structure mask:", structure_mask[:10])
print("structure mask intervals:", get_intervals(structure_mask))

From the structure mask intervals, we can see that

- the structure mask is `True` at residues 1-X and Y-Z. This means that the structure
  of these residues is *unspecified*.
- the structure mask is `False` at residues X-Y. This means that the structure of these
  residues is *specified*.

To ensure the design process focuses only on well-defined regions, we should remove
these residues with unspecified structures from our `target` object. However, we will
define the epitope first below, as it's slightly easier to identify the binding site
before the sequence is modified.

## Step 1.2: Specify the epitope

The epitope is the set of residues in the target that we want our designed binder to
bind to. We'll use the same residues as the RFdiffusion study as our epitope: the
residues at positions 62, 84, and 143.

NOTE: When using our API, residues are numbered starting from `1`, which follows the
canonical mmcif residue id system (`label_seq_id`), and not the author id system
(`auth_seq_id`). See the Appendix for more information.

To specify the epitope, we use the method `Protein.set_binding_at`. This method is used
to specify which residues are the *binding* residues i.e. the residues that should bind
to another protein chain. In this case, that other chain is the binder we're designing.

In [ ]:
binding_sites = [62, 84, 143]
target = target.set_binding_at(binding_sites)
# Verify binding is set ("B" is for binding, "N" for not-binding, and "U" for unknown)
target.get_binding_at(binding_sites)

### Remove residues with unspecified structure

Now let's remove the residues with unspecified structure. We can do this simply by
slicing the `target` protein using numpy-like slicing syntax, keeping only positions
where the structure mask is False.

In [ ]:
print("target length (old):", len(target))
target = target[~target.get_structure_mask()]
print("target length (new):", len(target))
print("structure mask (new):", get_intervals(structure_mask))

Note that by truncating the `target`, the positions of our binding site have changed.
This is why we chose to annotate the binding site first - so that we could simply use
the binding site positions aligned to the original target sequence. We can check the
positions of the new binding site by looking at the new binding array:

In [ ]:
binding = target.get_binding()
# NB: add one to get 1-indexed positions
binding_site = np.where(target.get_binding() == "B")[0] + 1
print("binding site:", binding_site)

### Visualize

It's always a good idea to visualize our `Protein`s to ensure that we've created them
correctly. First, let's define a helper function using the `molviewspec` package to
help us create the visualization. It's not necessary to understand the implementation
details.

In [142]:
@dataclass(frozen=True)
class ColorSpec:
    chain_id: str
    color: str
    positions: list[int] | None = None
    rep_type: RepresentationTypeT = "cartoon"


def visualize_cif(
    cif_string: str,
    colors: list[ColorSpec],
    rotation: list[list[int]] | None = None,
):
    builder = mvs.create_builder()
    model = (
        builder.download(url="structure.cif")
        .parse(format="mmcif")
        .model_structure()
        .transform(rotation=list(itertools.chain(*rotation)))
    )
    for color_spec in colors:
        component = model.component(
            selector=(
                mvs.ComponentExpression(label_asym_id=color_spec.chain_id)
                if color_spec.positions is None
                else [
                    mvs.ComponentExpression(
                        label_asym_id=color_spec.chain_id, label_seq_id=i
                    )
                    for i in color_spec.positions
                ]
            )
        )
        rep = component.representation(type=color_spec.rep_type)
        rep.color(color=color_spec.color)
    builder.molstar_notebook(
        data={"structure.cif": cif_string},
        width=600,
        height=500,
    )

Now, let's use the helper function, `visualize_cif`, to visualize our `target`.

In [ ]:
visualize_cif(
    cif_string=target.to_string(),
    colors=[
        # color the target chain a light blue
        ColorSpec(chain_id="A", color="#b5e2f5"),
        # color the epitope green, and use the ball_and_stick representation
        ColorSpec(
            chain_id="A",
            color="#6bb50a",
            positions=binding_sites,
            rep_type="ball_and_stick",
        ),
    ],
    # apply a rotation to make the binding site clearer
    rotation=[
        [0.8535534, -0.1464466, -0.5],
        [-0.1464466, 0.8535534, -0.5],
        [0.5, 0.5, 0.7071068],
    ],
)

<IPython.core.display.Javascript object>

The visualization shows the structure of IL-7Rα, and the epitope comprising three
residues spread across three adjacent loop regions, as expected!

## Step 1.3: Specify the binder length

For this tutorial, we'll generate binders of length `80`. Since the binder is what we're
designing, its sequence and structure is unknown. To represent this as a `Protein`
object, we simply need to create a `Protein` of length 80, whose sequence
and structure are both unspecified! The method `Protein.from_expr` allows us to do this
easily; this method constructs `Protein`s with unspecified sequence and structure from 
an **expr**ession describing its length. For our binder of length `80`, the expression
is simply `80`.

In [ ]:
binder = Protein.from_expr(80)
print("binder length:", len(binder))

We can inspect the `binder`'s sequence and structure mask to check that they are indeed
unspecified. Note that unknown residues in a sequence are represented as `X`.

In [ ]:
print("binder sequence:", binder.sequence)
binder_structure_mask_intervals = openprotein.sequence.get_intervals(
    binder.get_structure_mask()
)
print("binder structure mask:", binder_structure_mask_intervals)

Finally, now that we have our `target` and `binder`, we can combine them into a
`MolecularComplex` to form our query! We can do using the `MolecularComplex` constructor,
or more simply, by joining the two `Protein`s using an `&` like we do below. Chain ids
are assigned alphabetically, from left to right.

In [ ]:
query = target & binder
print("Query type", type(query))
print("Chains in query:", list(query.proteins.keys()))
print("Chain A (target):", query.proteins["A"].sequence)
print("Chain B (binder):", query.proteins["B"].sequence)

We can now use our `query` to generate binder designs.

# Step 2: Structure Generation

## Run RFdiffusion

To generate plausible structures for the binder using the RFdiffusion, we use the
function `session.models.rfdiffusion.generate`, passing in our `query`, desired number
of structures `N=100`, and additional any arguments to control the RFdiffusion algorithm.

In [ ]:
# Number of designs to generate
N = 100
rfdiffusion_job = session.models.rfdiffusion.generate(
    query=query,
    N=N,
    # Following Bennett et al. (2023), we reduce the noise added during
    # generation, which has been found to help with binder design, albeit at
    # the cost of some diversity.
    **{"denoiser.noise_scale_ca": 0.5, "denoiser.noise_scale_frame": 0.5},
)
rfdiffusion_job

RFdiffusionJob(job_id='3c61719e-67ea-4190-acdd-6e8e8aae7147', job_type='/models/rfdiffusion', status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2025, 12, 18, 21, 1, 46, 352527, tzinfo=TzInfo(0)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)

### Wait for completion

Wait for the job to complete. This will generally finish within an hour, depending on
server load.

In [12]:
rfdiffusion_job.wait_until_done(verbose=True, timeout=60*60)

Waiting: 100%|██████████████████████████████████████████████████| 100/100 [00:00<00:00, 600.93it/s, status=SUCCESS]


True

## Retrieve, inspect, and save generated structures

Here, we retrieve the generated structures, and inspect the first structure.

In [ ]:
generated_structures = rfdiffusion_job.get()
assert len(generated_structures) == N
first_structure = generated_structures[0]
print("chains in structure:", list(first_structure.proteins.keys()))
print("structure chain A sequence:", first_structure.proteins["A"].sequence)
print("structure chain B sequence:", first_structure.proteins["B"].sequence)
print("structure chain A mask:", first_structure.proteins["A"].get_structure_mask().any())
print("structure chain B mask:", first_structure.proteins["B"].get_structure_mask().any())

chains in design: ['A', 'B']
first design chain A sequence: b'GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG'
first design chain B sequence: b'DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIETKKFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDENKWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTP'
first design chain A mask: []
first design chain B mask: []


As we can see, we are returned with two chains with their structures
fully designed. Note however that RFdiffusion has re-positioned our
chains. Our binder is now chain A and the target is chain B. This is
important to be careful about. Also, RFdiffusion has set our binder
sequence as `G`, which is not a big deal, we will want to mask these for
inverse folding in our next step anyway.

We can also visually inspect the design:

In [ ]:
visualize_cif(first_structure)

<IPython.core.display.Javascript object>

Now let's iterate through these designs and save them.

In [ ]:
OUTPUT_DIR = Path("data/outputs/3DI3_binder_designs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for i in range(N):
    # Retrieve the completed design
    generated_structure = generated_structures[i]
    # Save the full complex
    with open(f"{OUTPUT_DIR}/design{i+1}.cif", "w") as f:
        f.write(generated_structure.to_string())

# Step 3: Sequence Design

Following Bennett et al. (2023), we'll use ProteinMPNN for inverse
folding to design sequences that adopt the designed binder structures.

For each of the 100 designs, we will generate 10 proposed sequences from
inverse folding.

In [ ]:
proteinmpnn_jobs = []
for i in range(N):
    generated_structure = generated_structures[i]
    # Mask the binder sequence to indicate that it should be generated
    generated_structure = generated_structure.mask_sequence(chain_ids="A") 

    # Use ProteinMPNN to design sequences for the binder backbone
    mpnn_job = session.models.proteinmpnn.generate(
        query=generated_structure,
        num_samples=10,
        temperature=0.1,  # Bennett et al. used low temperature
        seed=42, 
    )
    proteinmpnn_jobs.append(mpnn_job)

# Wait for all jobs to complete
for mpnn_job in proteinmpnn_jobs:
    mpnn_job.wait_until_done(timeout=600)
    assert mpnn_job.status == "SUCCESS"

Let's look at the output from one of the ProteinMPNN jobs.

In [ ]:
proteinmpnn_jobs[0].get()

[Score(name='generated-sequence-1', sequence='MKKTYTDTVRVIKTSPDTYSLSITVNLDGEKVTISMEVPNTKELTKKKTVTTSSGKKYKITLKLTLEGDEWKVEITIEEL', score=array([1.1707])),
 Score(name='generated-sequence-2', sequence='MTKKETTTAKAIEISPDTLDIVIYVNLNGETVTLAMTIPNTPKLKKKVTVTTSSGKKYEIDLEITLEGDEYKINITIKEL', score=array([1.1413])),
 Score(name='generated-sequence-3', sequence='MKKEEKTTAKAIKISPDTYEISIDIELDGEKVTISKTIPNTEELEKEVTVTTSSGKKYKIKLKLKLKGDEWEIEITIEEL', score=array([1.1164])),
 Score(name='generated-sequence-4', sequence='MTKTETTYVKAIEVSPDTLQAVLDITLDGEKVTLALTIPNTKEFTKEKTVTTSSGKKYKITLKGTLEGDKLKVTITIEEL', score=array([1.0946])),
 Score(name='generated-sequence-5', sequence='MTKTYTTTVRVIEISPNTLDYVLYVNLNGETVVIAKTIPNTPEFTHHDIVTTSSGKKYEIDIKGKLEGDNLNLKITIKEL', score=array([1.2136])),
 Score(name='generated-sequence-6', sequence='ATTTETTRARAIKISPDKYEISIDLTLNGETVTLNLVIPNTPTLTVTRTVTTSSGKKYKVTLKLTLEGDEWLIDITTEEL', score=array([1.15])),
 Score(name='generated-sequence-7', sequence='ETSKEHTTARAIQIDPTTYDTVIDITLGGEKQTIAMRV

Each of these 10 sequences correspond to the first design from
RFdiffusion. Let's save the ProteinMPNN predictions together with the
RFdiffusion designs so that we have a 1000 of these potential designs.

In [ ]:
scores = []
for i in range(N):
    generated_structure = generated_structures[i]
    mpnn_job = proteinmpnn_jobs[i]
    mpnn_results = mpnn_job.get()
    for j, (_, sequence, score) in enumerate(mpnn_results):
        # replace chain explicitly due to defensive copy
        binder = generated_structure.proteins["A"] 
        binder.sequence = sequence
        generated_structure.proteins["A"] = binder 
        scores.append(score.item())
        with open(f"{OUTPUT_DIR}/design{i+1}_mpnn{j+1}.cif", "w") as f:
            f.write(generated_structure.to_string())
with open(f"{OUTPUT_DIR}/mpnn_scores.txt", "w") as f:
    f.write("\n".join([str(score) for score in scores]))

Let's just verify that our new designed model looks correct:

In [19]:
from openprotein import Model

OUTPUT_DIR = Path("data/outputs/3DI3_binder_designs")

proteinmpnn_model = Model.from_filepath(f"{OUTPUT_DIR}/design1_mpnn1.pdb")
print("chains in proteinmpnn + rfdiffusion design:", list(proteinmpnn_model.proteins.keys()))
print("binder sequence:", proteinmpnn_model.proteins["A"].sequence)
print("target sequence:", proteinmpnn_model.proteins["B"].sequence)
print("binder mask:", proteinmpnn_model.proteins["A"].get_structure_mask())
print("target mask:", proteinmpnn_model.proteins["B"].get_structure_mask())

Notice that what we have is a combination of the two models: the binder
structure is from RFdiffusion and the inverse folded binder sequence is
from ProteinMPNN. The next step is to check if the predicted multimer
folds into what we expect.

# Step 4: In Silico Validation

Whilst Bennett et al. (2023) and Watson et al. (2023) both used
AlphaFold2 to re-fold their designs, we will use ESMFold instead to
validate our designs.

The key insight from their paper is that AF2's prediction confidence
metrics (particularly pAE<sub>interaction</sub>) can effectively
discriminate successful binders from failures. Bennett et al. found that
the pAE<sub>interaction</sub> metric (average pAE of interchain residue
pairs) was extremely effective at identifying successful binders, with
sharp increases in success rates for designs with
pAE<sub>interaction</sub> \< 10.

We can obtain these same metrics with ESMFold, which will also run a lot
faster than AF2. The papers also used AF2 initial guess, with templates,
which are features not yet ready for use with our AF2 on our platform.
This walkthrough will be updated if add support for these features and
find that the AF2 metrics perform better. The key point to note is that
our platform allows easy drop-in replacements for various steps in your
protein design pipeline.

In [20]:
proteinmpnn_models = []
for i in range(N):
    for j in range(10):
        proteinmpnn_model = Model.from_filepath(f"{OUTPUT_DIR}/design{i+1}_mpnn{j+1}.pdb")
        proteinmpnn_models.append(proteinmpnn_model)
esmfold_job = session.fold.esmfold.fold(
   proteinmpnn_models
)
esmfold_job

FoldJob(num_records=1000, job_id='58a6a0ee-285a-40ff-bc18-32392d9f4d51', job_type=<JobType.embeddings_fold: '/embeddings/fold'>, status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2025, 12, 19, 19, 1, 16, 12126, tzinfo=TzInfo(0)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)

Wait for completion. This will likely take quite around an hour.

In [21]:
esmfold_job.wait_until_done(verbose=True, timeout=60*60)

Waiting: 100%|██████████████████████████████████████████████████| 100/100 [00:00<00:00, 566.69it/s, status=SUCCESS]


True

Let's retrieve and inspect the ESMFold fold results:

In [22]:
esmfold_results = esmfold_job.get()
esmfold_seq, esmfold_model = esmfold_results[0] # a fold returns (seq, model) tuples 
print("chains in folded model:", list(esmfold_model.proteins.keys()))
print("first fold chain A sequence:", esmfold_model.proteins["A"].sequence)
print("first fold chain B sequence:", esmfold_model.proteins["B"].sequence)
print("first fold chain A mask:", esmfold_model.proteins["A"].get_structure_mask())
print("first fold chain B mask:", esmfold_model.proteins["B"].get_structure_mask())

chains in folded model: ['A', 'B']
first fold chain A sequence: b'MKKTYTDTVRVIKTSPDTYSLSITVNLDGEKVTISMEVPNTKELTKKKTVTTSSGKKYKITLKLTLEGDEWKVEITIEEL'
first fold chain B sequence: b'DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIETKKFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDENKWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTP'
first fold chain A mask: []
first fold chain B mask: []


As expected, our sequences are the same and the structure mask is there,
meaning the whole structure for the complex is predicted by ESMFold.

We can also retrieve the pAE matrix predicted by ESMFold,
which are useful as metrics for measuring the success of our designs. This could take awhile to retrieve all the results.

In [23]:
esmfold_pae_results = esmfold_job.pae
esmfold_seq, esmfold_complex_pae = esmfold_pae_results[0]
print("pae interaction shape:", esmfold_complex_pae.shape)

pae interaction shape: (273, 273)


## Ranking designs by metrics

Following Bennet et al. (2023), we'll rank our designs based on:

1.  Monomer pLDDT (confidence that sequence folds to designed structure)
2.  Complex pAE interaction (confidence that binder forms
    intended interface)
3.  Complex Cα RMSD to designed structure

In [24]:
import pandas as pd 

design_files = []
plddt_scores = []
pae_scores = []
rmsd_scores = []
for i in range(N*10):
    # Get ESMFold predictions
    _, esmfold_model = esmfold_results[i]

    design_files.append(f"design{i//10}_mpnn{i%10}.pdb")

    binder = esmfold_model.proteins["A"]
    target = esmfold_model.proteins["B"]

    # Get pLDDT of binder
    plddt_score = np.mean(binder.plddt)
    plddt_scores.append(plddt_score)

    # Get pAE
    _, esmfold_complex_pae = esmfold_pae_results[i] 
    binder_target_pae = esmfold_complex_pae.squeeze() # squeeze the shape
    pae_interaction_1 = np.mean(binder_target_pae[len(binder):,:len(binder)])
    pae_interaction_2 = np.mean(binder_target_pae[:len(binder),len(binder):])
    pae_interaction_total = (pae_interaction_1 + pae_interaction_2) / 2 
    pae_scores.append(pae_interaction_total)

    # RMSD between designed binder and folded binder
    designed_binder = rfdiffusion_models[i//10].proteins["A"]
    folded_binder = binder

    binder_rmsd = designed_binder.rmsd(folded_binder, backbone_only=True)
    rmsd_scores.append(binder_rmsd)

df = pd.DataFrame({"design_file": design_files, "plddt": plddt_scores, "pae": pae_scores, "rmsd": rmsd_scores})
print(df.head(10))

         design_file      plddt        pae      rmsd
0  design0_mpnn0.pdb  64.985374  26.108816  2.508876
1  design0_mpnn1.pdb  60.023247  26.559431  2.313612
2  design0_mpnn2.pdb  64.042992  26.312916  3.289286
3  design0_mpnn3.pdb  65.501999  25.959415  2.709022
4  design0_mpnn4.pdb  65.479004  24.624670  2.541036
5  design0_mpnn5.pdb  72.557373  25.386769  0.839039
6  design0_mpnn6.pdb  58.761375  25.435695  3.850175
7  design0_mpnn7.pdb  69.294373  24.323982  1.071133
8  design0_mpnn8.pdb  63.944874  26.111162  3.166059
9  design0_mpnn9.pdb  61.955128  26.144578  4.354014


# Analysis and Ranking

Let's rank the successful designs by their AF2 metrics:

In [25]:
import pandas as pd

df_sorted = df.sort_values(by=["plddt", "pae", "rmsd"], ascending=[False, True, True])

print(df_sorted.head(10))

# Save rankings
df_sorted.to_csv(OUTPUT_DIR / f"rankings.csv", index=False)

            design_file      plddt        pae      rmsd
993  design99_mpnn3.pdb  85.635620  14.546918  0.576264
362  design36_mpnn2.pdb  85.087120  24.013505  0.558198
365  design36_mpnn5.pdb  84.757126  11.708263  0.376451
363  design36_mpnn3.pdb  83.749001  14.447840  0.695756
361  design36_mpnn1.pdb  83.108627  15.472516  0.395722
469  design46_mpnn9.pdb  82.964622  11.207534  0.656172
387  design38_mpnn7.pdb  82.905998  23.893167  0.725844
280  design28_mpnn0.pdb  82.414253  24.409521  0.538304
895  design89_mpnn5.pdb  82.379875  13.552719  0.630388
385  design38_mpnn5.pdb  82.042374  25.747786  0.546411


# Summary

In this tutorial, we've demonstrated the deep learning-augmented binder
design workflow using RFdiffusion, ProteinMPNN and ESMFold:

1.  **Target Selection**: Downloaded 3DI3 structure from RCSB PDB
2.  **Hotspot Identification**: Selected binding regions based on known
    ligand-receptor interactions
3.  **Structure Generation**: Used RFdiffusion to generate binder
    backbones
4.  **Sequence Design**: Applied ProteinMPNN for fast, efficient
    sequence design
5.  **Validation**: Used ESMFold to rank designs based on:
    - Monomer folding confidence (pLDDT)
    - Complex formation confidence (pAE interaction)
    - Structural accuracy (RMSD)

This approach achieves ~10-fold higher success rates compared to purely
physics-based methods by leveraging deep learning models to identify
Type I failures (sequences that don't fold as intended) and Type II
failures (structures that don't bind as intended).

## Next Steps

The top-ranked designs from this workflow can be:

1.  Expressed and purified for experimental validation
2.  Tested for binding affinity
3.  Further optimized through additional rounds of design